# An Alarm Score Is Not a Probability

This executable companion uses strictly synthetic Gaussian scores. Pure label shift preserves class-conditional scores but changes posterior probabilities and operational costs. See the README and manuscript in this directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
from analysis import (Scenario, posterior, correct_prior, operating_rates,
                      precision_at_prior, cost_probability_threshold,
                      score_threshold, decision_operating_point, run_study)
scenario = Scenario()
print(scenario)

In [ ]:
# Ranking is unchanged by the exact prior-odds correction.
scores = np.array([-1.0, 0.0, 1.0, 2.0, 3.0])
p_source = posterior(scores, scenario.source_prior, scenario.separation)
p_adjusted = correct_prior(p_source, scenario.source_prior, scenario.target_prior)
assert np.allclose(p_adjusted, posterior(scores, scenario.target_prior, scenario.separation))
assert np.array_equal(np.argsort(p_source), np.argsort(p_adjusted))
print('Development probabilities:', np.round(p_source, 4))
print('Deployment probabilities: ', np.round(p_adjusted, 4))

In [ ]:
# Identical detector rates need not imply identical precision.
tpr, fpr = operating_rates(1.5, scenario.separation)
for pi in (scenario.source_prior, scenario.target_prior):
    print(f'Prevalence {pi:.1%}: precision {float(precision_at_prior(pi, tpr, fpr)):.1%}')
cost_prob = cost_probability_threshold(scenario.false_alarm_cost, scenario.missed_event_cost)
frozen = score_threshold(cost_prob, scenario.source_prior, scenario.separation)
updated = score_threshold(cost_prob, scenario.target_prior, scenario.separation)
for name, cutoff in [('frozen', frozen), ('updated', updated)]:
    result = decision_operating_point(cutoff, scenario.target_prior, scenario.separation,
                                      scenario.false_alarm_cost, scenario.missed_event_cost)
    print(name, 'threshold', round(cutoff, 3), 'expected cost', round(result['cost_per_case'], 3))

In [ ]:
# Regenerate the small study in a temporary directory; tracked evidence is untouched.
with TemporaryDirectory() as folder:
    temporary = Path(folder)
    summary = run_study(quick=True, output_dir=temporary)
    assert len(list((temporary / 'results').glob('*.csv'))) == 10
    assert len(list((temporary / 'figures').glob('*.png'))) == 8
    assert summary['target_brier_adjusted'] < summary['target_brier_frozen']
    print('AUC:', round(summary['analytic_roc_auc'], 3),
          '| cost ratio:', round(summary['frozen_cost_ratio'], 2))

## Interpretation

The Bayes adjustment is valid only if the class-conditional score distributions remain unchanged and the source probabilities are calibrated. The manuscript also tests violations of those assumptions and outcome-verification selection bias. The synthetic costs are illustrative, not real savings.